In [ ]:
include( "../model.jl" )
include( "../prob.jl" )

using ThreadsX
using StatsBase
using Printf

using StatsPlots
using Distributions

home, base = expfolder( "system" )

In [ ]:
# System parameters.
N = 128

# Import noise, speed, and density lists.
include( "../0-params/nos-list.jl" )
include( "size-0-params.jl" )

# Color map.
# colormap = cgrad( [:black, :gray, :white, :mediumpurple], [0.0, 0.10, 0.40, 0.50, 0.75, 1.0] )
colormap = cgrad( :ice )

In [ ]:
ρ = ρlist[1]
println( "Densities: ", (ρ, ρlist) )

# Initialize folder names.
folderdata = [findfolder( N, ρ, slist[i], ηlist[k]; base=base ) for i ∈ 1:Ns, k ∈ 1:Nη];

In [ ]:
# Initialize progress meter.
meter = Progress( Ns, "Extracting critical noise information." )

# Initialize mean order and critical noise matrix.
μφdata = Matrix{defFloat}( undef, Ns, Nη )
φdatalist = Matrix{Vector{defFloat}}( undef, Ns, Nη )

# Iterate through the data.
T = 100_000
@threads for i ∈ 1:Ns
    # Import order parameter data.
    for k ∈ 1:Nη
        φdatalist[i,k] = vec( readdlm( folderdata[i,k]*"order_$(T)-end.txt" ) )
    end

    # Compute the mean order parameter.
    μφdata[i,:] = mean.( φdatalist[i,:] )

    # Update progress meter.
    next!( meter )
end

# Extract the critical noise for each (ρ,s) pair.
ηclist = [criticalnoise( ηlist, μφdata[i,:] ) for i ∈ 1:Ns];

In [ ]:
# Plot an example order-disorder trend.
plt = plot(; size=(400,250), legend=false )

for i ∈ 1:Ns
    plot!( plt, ηlist, μφdata[i,:]; color=colormap[(i-1)/(Ns-1)], marker=:circ, lw=2 )
end

plot!( plt; xlims=(0,ηlist[end]), ylims=(0,1) )
plot!( plt; xlabel="noise amplitude, "*L"η", ylabel="mean order, "*L"⟨φ⟩" )

In [ ]:
# Save data to pass to the HPC.
if true
    for i ∈ 1:Ns
        folder = folderdata[i,1]*"../"
        writedlm( folder*"order-0.5.txt", ηclist[i] )
    end
end